In [8]:
import gymnasium as gym
from gymnasium.spaces import Discrete, Box
import numpy as np
from typing import Optional, Any

In [ ]:
class ContinuousStateDiscreteAction(gym.Env):
    """Пример среды: непрерывные состояния, дискретные действия"""

    metadata = {"render_modes": ["human"], "render_fps": 4}

    def __init__(
        self,
        *,
        render_mode: Optional[str] = None,
        seed: Optional[int] = None,
        max_steps: int = 10,
    ) -> None:
        super().__init__()

        # Непрерывное пространство состояний
        self.observation_space = Box(
            low=np.array([-1.0], dtype=np.float32),
            high=np.array([1.0], dtype=np.float32),
            dtype=np.float32,
        )
        # Дискретное пространство действий
        self.action_space = Discrete(3)

        # Семантические описания (удобно для отладки)
        self.action_in_words = ["уменьшить", "ничего", "увеличить"]

        # Инициализация
        self.state = np.array([0.0], dtype=np.float32)
        self.step_count = 0
        
        # Параметры визуализации
        self.render_mode = render_mode
        
        # Инициализация случайности
        self.np_random, _ = gym.utils.seeding.np_random(seed)

        self.max_steps = max_steps

    # -----------------------
    # Основные методы Gym API
    # -----------------------

    def reset(
        self,
        *,
        seed: Optional[int] = None,
        options: Optional[dict[str, Any]] = None,
    ):
        super().reset(seed=seed)
        self.state = self.np_random.uniform(low=-0.5, high=0.5, size=(1,)).astype(np.float32)
        self.step_count = 0
        info = {}

        if self.render_mode == "human":
            print(f"[reset] начальное состояние: {self.state[0]:.3f}")

        return self.state, info

    def step(self, action: int):
        # Проверка допустимости действия
        assert self.action_space.contains(action), f"Неверное действие: {action}"

        # Динамика
        delta = {-1: -0.1, 0: 0.0, 1: 0.1}[action - 1]  # 0,1,2 → -1,0,1
        noise = self.np_random.normal(0, 0.01)
        self.state = np.clip(self.state + delta + noise, -1.0, 1.0)

        # Вознаграждение — чем ближе к 0, тем лучше
        reward = -abs(self.state[0])

        self.step_count += 1

        # 👇 Termination по числу шагов
        terminated = self.step_count >= self.max_steps
        truncated = False   # Можно добавить ограничение по времени
        info = {}

        if self.render_mode == "human":
            print(
                f"[step {self.step_count:02d}] "
                f"action={self.action_in_words[action]:<10} "
                f"state={self.state[0]:+.3f} "
                f"reward={reward:+.3f}"
            )

        return self.state, reward, terminated, truncated, info

    def render(self):
        print(f"Текущее состояние: {self.state[0]:+.3f}")

    def close(self):
        pass

In [10]:
if __name__ == "__main__":
    env = ContinuousStateDiscreteAction(render_mode="human", max_steps=10)
    obs, info = env.reset()

    done = False
    while not done:
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

    env.close()

[reset] начальное состояние: -0.412
[step 01] action=уменьшить  state=-0.508 reward=-0.508
[step 02] action=уменьшить  state=-0.590 reward=-0.590
[step 03] action=уменьшить  state=-0.670 reward=-0.670
[step 04] action=увеличить  state=-0.551 reward=-0.551
[step 05] action=увеличить  state=-0.449 reward=-0.449
[step 06] action=ничего     state=-0.443 reward=-0.443
[step 07] action=увеличить  state=-0.356 reward=-0.356
[step 08] action=увеличить  state=-0.250 reward=-0.250
[step 09] action=увеличить  state=-0.142 reward=-0.142
[step 10] action=увеличить  state=-0.052 reward=-0.052
